# Sweep AN1 inhibition

This simulation is slow. Run this notebook only to regenerate the supplied numerical sweep. New results are saved under `results/`; Figure 5 reads the supplied archive in `saved_results/`.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / 'src'))
RESULTS = ROOT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
plt.style.use(ROOT / 'notebooks/ncb.mplstyle')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotting as p
import matplotlib as mpl

from metpy.interpolate import interpolate_to_points
import jax.numpy as jnp

from Gbim_import_dat import get_Gbim_data_chirpfield, get_Gbim_data_DC
from stim_gen import makeCHIRPstim
import scipy
from models.fix import run_model, Params
from collections import defaultdict
from tqdm import tqdm




MAX = 40
step = 10
cpauMax = 1000 + MAX
max_len = cpauMax

In [ ]:
[CDUR, CPAU, CPER, DCs, NPUL, rXY]= get_Gbim_data_chirpfield()
[CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]=get_Gbim_data_DC()

keys = ['CDUR', 'CPAU', 'CPER', 'CDC', 'NPUL', 'rXY']
data_cpf = dict(zip(keys, [CDUR, CPAU, CPER, DCs, NPUL, rXY]))
data_dc = dict(zip(keys, [CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]))

data_all = {key: np.concatenate((data_cpf[key], data_dc[key])) for key in keys}

In [ ]:
duri0 = paui0 = np.arange(0, max_len, step)
duri, paui = np.meshgrid(duri0,paui0)

CDURinterp=np.concatenate((CDUR,[0, 0]))
CPAUinterp=np.concatenate((CPAU,[0, 1000]))
rXYinterp=np.concatenate((rXY,[0, 0]))

points = np.array([CDURinterp + np.random.randn(*CDURinterp.shape)/1_000, CPAUinterp+ np.random.randn(*CPAUinterp.shape)/1_000]).T
new_points = np.array([duri, paui]).T.reshape((-1, 2))

cpf_behave = interpolate_to_points(points, np.clip(rXYinterp, 0, np.inf), new_points, interp_type='natural_neighbor', minimum_neighbors=4, rbf_smooth=0)
cpf_behave = np.maximum(cpf_behave, 0)
cpf_behave = cpf_behave.reshape((len(duri), len(paui))).T

plt.pcolor(duri, paui, cpf_behave, cmap='Greys')
plt.plot(CDUR, CPAU, 'o')
plt.xlim(0)
plt.ylim(0)
plt.show()

In [ ]:
input_stim, s, cpau, cdur = makeCHIRPstim(
    cpauMax=1000 + MAX,
    cdurMax=1000 + MAX,
    cpauMin=MAX,
    cdurMin=MAX,
    ppau=20,
    pdur=20,
    Fs=1000,
    num_steps=32,
    stim_len=10_000,

)
input_stim = jnp.array(input_stim)
points = np.array(
    [
        s["cpau"] + np.random.randn(*s["cpau"].shape) / 100,
        s["cdur"] + np.random.randn(*s["cdur"].shape) / 100,
    ]
).T

xi0 = yi0 = np.arange(0, cpauMax, step)
xi, yi = np.meshgrid(xi0, yi0)
new_points = np.array([xi, yi]).T.reshape((-1, 2))

cduri = xi.copy()
cpaui = yi.copy()

In [ ]:
# inh_mults = [0, 0.125, 0.25, 0.5, 1, 2, 4]#, 12]
# inh_mults = [0, 0.125, 0.25, 0.5, 0.75, 1, 1.25, 1.5, 2, 4]
mults = np.logspace(-4, 2, base=2, num=19)
print(mults)

resp_ln4 = {}
model_outputs = {}
cpf = {}
for mult in tqdm(mults):
    param = Params()
    param.an1_filter_inh_gain *= mult

    key = mult
    model_output = run_model(input_stim, param, verbose_output=False, return_dict=True)
    # model_outputs[key] = model_output
    resp_ln4[key] = np.nanmean(model_output['ln4'], axis=0)
    cpf[key] = interpolate_to_points(points, resp_ln4[key], new_points, interp_type='rbf', rbf_func='linear', rbf_smooth=20)#, minimum_neighbors=1)
    cpf[key] = cpf[key].reshape((len(xi), len(yi)))

resp_ln4['stim'] = np.mean(input_stim, axis=0)
cpf['stim'] = interpolate_to_points(points, resp_ln4['stim'], new_points, interp_type='rbf', rbf_func='linear', rbf_smooth=20)#, minimum_neighbors=1)
cpf['stim'] = cpf['stim'].reshape((len(xi), len(yi)))

cpf['behavior'] = cpf_behave

for key, resp in resp_ln4.items():
    s[f'rXY_{key}'] = resp

# Source notebook cell 7: required sweep keys (exploratory plot omitted).
keys = mults

In [ ]:
def transect_slice(cper, cdc, cper_low, cper_high, ppf):
    """Estimate a smoothed transect slice from filtered data.
    """
    idx = np.where(
        np.logical_and(
            cper > cper_low, cper < cper_high
        )
    )[0]
    x, y = cdc[idx], ppf[idx]

    # x = np.insert(x, [0, 0, 0], [0, 0.001, 0.002])
    # y = np.insert(y, [0, 0, 0], 0)

    # Fix duplicate x vals by avg y over all x with same val
    xm = np.unique(x)
    y = np.array([np.nanmax(y[x == xi]) for xi in xm])
    x = xm
    # sort
    sort_idx = np.argsort(x)
    x, y = x[sort_idx], y[sort_idx]
    # smooth
    x = x[~np.isnan(y)]
    y = y[~np.isnan(y)]

    bins = np.percentile(x, np.linspace(0, 100, 32))
    ys, xs, _ = scipy.stats.binned_statistic(x, y, bins=bins, range=(0, 1))
    xs = xs - np.diff(xs, prepend=0) / 2

    # ypad = np.pad(y, (3, 3), mode="edge")  # pad repeat edge values
    # # ypad = np.pad(y, (1, 1), mode="constant", constant_values=np.nan)  # pad repeat edge values
    # ys = np.convolve(ypad, np.ones(7) / 7, mode="valid")  # 3-val running avg
    return x, y, ys, xs

In [ ]:
cper_low = 200
cper_high = 500
# cdc_low = defaultdict(lambda: 0.33, {0: 0.9, 1: 0.9, 2: 0.9, 4: 0.6})
# cdc_high = defaultdict(lambda: 0.67, {0: 1.0, 1: 1.0, 2: 1.0, 4: 0.8})
cdc_low = defaultdict(lambda: 0.33)
cdc_high = defaultdict(lambda: 0.67)

cper=duri+paui
cdc=duri/(cper)

best_cdc = []
max_cdc_resp = []
best_cper = []
max_cper_resp = []

plt.figure(figsize=(18, 4))
for cnt, key in enumerate(keys):
    plt.subplot(3, len(keys), 1+cnt)
    im = plt.pcolormesh(duri, paui, cpf[key] / np.percentile(cpf[key], 99), cmap='Greys')
    plt.xlim(0, 600)
    plt.ylim(0, 600)
    plt.clim(0, 1)

    x, y, ys, xs = transect_slice(cper.flatten(), cdc.flatten(), cper_low, cper_high, np.clip(cpf[key].flatten(), 0, 1))
    ys = np.insert(ys, 0, 0)
    ys = p.nmax(ys)
    best = np.argmax(ys)

    plt.subplot(3, len(keys), 1+cnt + len(keys))
    # plt.plot(x, y, 'ok', alpha=0.05)
    plt.plot(xs, ys, 'k', alpha=0.8)
    plt.plot(xs[best], ys[best], 'o', c='limegreen')
    plt.plot(xs[-1], ys[-1], 'o', c='limegreen')
    plt.xlim(0, 1)
    plt.ylim(0, 1.1)

    best_cdc.append(xs[best])
    max_cdc_resp.append(ys[-1] / ys[best])


    x, y, ys, xs = transect_slice(cdc.flatten(), cper.flatten(), cdc_low[key], cdc_high[key], np.clip(cpf[key].flatten(), 0, 1))
    ys = np.insert(ys, 0, 0)
    ys = p.nmax(ys)
    best = np.argmax(ys)

    plt.subplot(3, len(keys), 1+cnt + 2*len(keys))
    # plt.plot(x, y, 'ok', alpha=0.05)
    plt.plot(xs, ys, 'k', alpha=0.8)
    plt.plot(xs[best], ys[best], 'o', c='limegreen')
    plt.plot(xs[-1], ys[-1], 'o', c='limegreen')
    plt.xlim(0, 1.05 * np.max(x))
    plt.ylim(0, 1.1)
    # plt.xticks([])
    # plt.xticks([])

    best_cper.append(xs[best])
    max_cper_resp.append(ys[-1] / ys[best])
    # plt.ylabel(species)

max_cdc_resp = np.clip(max_cdc_resp, 0, 1)
max_cper_resp = np.clip(max_cper_resp, 0, 1)

# plt.tight_layout()
plt.xlabel('CDC')
plt.show()

In [ ]:
cper_low = 200
cper_high = 500
# cdc_low = defaultdict(lambda: 0.33, {0: 0.9, 1: 0.9, 2: 0.9, 4: 0.6})
# cdc_high = defaultdict(lambda: 0.67, {0: 1.0, 1: 1.0, 2: 1.0, 4: 0.8})
cdc_low = defaultdict(lambda: 0.33)
cdc_high = defaultdict(lambda: 0.67)

cper=duri+paui
cdc=duri/(cper)

best_cdc = []
max_cdc_resp = []
best_cper = []
max_cper_resp = []

cdc_transect_x = []
cdc_transect_y = []
cper_transect_x = []
cper_transect_y = []


n = len(keys)
blue_colors = mpl.colors.LinearSegmentedColormap.from_list(
    "myblues",
    ["#dceaf6", "#8fb9dd", "#3f7fba", "#0b559f", "#08306b"]
)(np.linspace(0, 1, n))

# red_colors = mpl.colors.LinearSegmentedColormap.from_list(
#     "myreds",
#     ["#f7d8d8", "#ef9a9a", "#d85757", "#b22222", "#67001f"]
# )(np.linspace(0, 1, n))

# # define the colormap once

red_cmap = mpl.colors.LinearSegmentedColormap.from_list(
    "myreds",
    ["#f7d8d8", "#ef9a9a", "#d85757", "#b22222", "#67001f"]
)

param = np.asarray(np.log2(keys))  # one value per key
norm = mpl.colors.Normalize(param.min(), param.max())
red_colors = red_cmap(norm(param))


plt.figure(figsize=(6, 3))
for cnt, key in enumerate(keys):

    x, y, ys, xs = transect_slice(cper.flatten(), cdc.flatten(), cper_low, cper_high, np.clip(cpf[key].flatten(), 0, 1))
    ys = np.insert(ys, 0, 0)
    ys = p.nmax(ys)
    best = np.argmax(ys)

    plt.subplot(121)
    # plt.plot(x, y, 'ok', alpha=0.05)
    plt.plot(xs, ys, alpha=0.8, c=red_colors[cnt])
    plt.plot(xs[best], ys[best], 'o', c=red_colors[cnt])
    plt.plot(xs[-1], ys[-1], 's', c=red_colors[cnt])
    plt.xlim(0, 1)
    plt.ylim(0, 1.1)
    plt.xticks([0, 0.5, 1])
    plt.yticks([0, 1])
    plt.xlabel('Chirp duty cycle')
    plt.ylabel('Normalized response')

    best_cdc.append(xs[best])
    max_cdc_resp.append(ys[-1] / ys[best])
    cdc_transect_x.append(xs)
    cdc_transect_y.append(ys)

    x, y, ys, xs = transect_slice(cdc.flatten(), cper.flatten(), cdc_low[key], cdc_high[key], np.clip(cpf[key].flatten(), 0, 1))
    x = x / 1000
    xs = xs / 1000
    ys = np.insert(ys, 0, 0)
    ys = p.nmax(ys)
    best = np.argmax(ys)

    plt.subplot(122)
    # plt.plot(x, y, 'ok', alpha=0.05)
    plt.plot(xs, ys, alpha=0.8, c=red_colors[cnt])
    plt.plot(xs[best], ys[best], 'o', c=red_colors[cnt])
    plt.plot(xs[-1], ys[-1], 's', c=red_colors[cnt], label=key)
    plt.xlim(0, 1.05 * np.max(x))
    plt.ylim(0, 1.1)
    plt.xticks([0, 1, 2])
    plt.yticks([0, 1])
    best_cper.append(xs[best])
    max_cper_resp.append(ys[-1] / ys[best])
    cper_transect_x.append(xs)
    cper_transect_y.append(ys)


    # plt.ylabel(species)
    plt.xlabel('Chirp period [s]')

max_cdc_resp = np.clip(max_cdc_resp, 0, 1)
max_cper_resp = np.clip(max_cper_resp, 0, 1)

sm = mpl.cm.ScalarMappable(norm=norm, cmap=red_cmap)
sm.set_array([])
fig = plt.gcf()
cbar = fig.colorbar(sm, ax=fig.axes, fraction=0.04, pad=0.02, shrink=0.5)
cbar.set_label("Inhibition strength (fold change)")      # or whatever the colors represent
cbar.set_ticks(np.arange(-4, 2.1, 2))
cbar.set_ticklabels(2.0**np.arange(-4, 2.1, 2))
plt.show()

In [ ]:
results = {'best_cdc': np.array(best_cdc), 'best_cper': np.array(best_cper),
           'inh_mults': mults,
           'max_cdc_resp': np.array(max_cdc_resp), 'max_cper_resp': np.array(max_cper_resp),
           'cdc_transect_x': np.array(cdc_transect_x),
           'cdc_transect_y': np.array(cdc_transect_y),
           'cper_transect_x': np.array(cper_transect_x),
           'cper_transect_y': np.array(cper_transect_y),

           }
np.savez(RESULTS / 'results_model_AN1inh.npz', **results)